# WEEK 3 ASSIGNMENT

# STEP 1:  Libraries Import

In [1]:
import sqlite3
import pandas as pd

print("All libraries imported successfully")

All libraries imported successfully


# STEP 2: Database Connected

In [2]:
conn = sqlite3.connect('sales_analysis')
cursor = conn.cursor()

# Step 3: Setup Data

## STEP 3.1: Load CSV File

In [3]:
data = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

# Creating table superstore
data.to_sql("superstore_raw", conn, if_exists="replace", index=False)
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


# STEP 3.2: Creating Tables

## Table 1: Customers Table

In [4]:
cursor.execute('''CREATE TABLE customers AS
SELECT DISTINCT
    "Customer ID" AS customer_id,
    "Customer Name" AS customer_name,
    Segment
FROM superstore_raw;''')

conn.commit()                                    
print("Customers Table Created!")

Customers Table Created!


# Table 2: Products Table

In [5]:
cursor.execute('''CREATE TABLE products AS
SELECT DISTINCT
    "Product ID" AS product_id,
    "Product Name" AS product_name,
    Category,
    "Sub-Category" AS sub_category
FROM superstore_raw''')

conn.commit()                                    
print("Products Table Created!")

Products Table Created!


# Table 3: Orders Table

In [6]:
cursor.execute('''CREATE TABLE orders AS
SELECT DISTINCT
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    "Ship Date" AS ship_date,
    "Ship Mode" AS ship_mode,
    "Customer ID" AS customer_id,
    "Product ID" AS product_id,
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw''')

conn.commit()
print("Orders Table Created!")

Orders Table Created!


# Helper Function

In [7]:
def run_query(query):
   return pd.read_sql_query(query,conn)

# Step 4: Perform Required Queries 

## Q1. Find all orders where sales are greater than the average sales. (Subquery)  

In [11]:
q1 = "select order_id, order_date, Sales from orders where sales > (select avg(Sales) from orders)"
df_q1 = run_query(q1)
df_q1

,order_id,order_date,Sales
0,CA-2016-152156,11/8/2016,261.9600
1,CA-2016-152156,11/8/2016,731.9400
2,US-2015-108966,10/11/2015,957.5775
3,CA-2014-115812,6/9/2014,907.1520
4,CA-2014-115812,6/9/2014,1706.1840
...,...,...,...
2354,US-2016-103674,12/6/2016,271.9600
2355,US-2016-103674,12/6/2016,249.5840
2356,US-2016-103674,12/6/2016,437.4720
2357,CA-2017-121258,2/26/2017,258.5760


In [64]:
print("""BRIEF INSIGHTS!! -- Orders with Sales Greater Than Average

These orders generate more sales than a typical order and contribute significantly to revenue.""")

BRIEF INSIGHTS!! -- Orders with Sales Greater Than Average

These orders generate more sales than a typical order and contribute significantly to revenue.


## Q2. Find the highest sales order for each customer. (Subquery)  

In [12]:
q2 = "select * from orders o where Sales = (select max(Sales) from orders where customer_id = o.customer_id)"
df_q2 = run_query(q2)
df_q2

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
1,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
2,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
3,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,FUR-TA-10000577,1044.6300,3,0.00,240.2649
4,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,FUR-BO-10004834,3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...
790,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,OFF-BI-10003656,1087.9360,8,0.20,353.5792
791,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,TEC-CO-10003763,2799.9600,5,0.20,944.9865
792,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,OFF-AP-10002945,2405.2000,8,0.00,793.7160
793,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,OFF-BI-10004600,735.9800,2,0.00,331.1910


In [63]:
print("""BRIEF INSIGHTS!! -- Highest Sales Order for Each Customer

Shows the biggest purchase made by each customer.""")

BRIEF INSIGHTS!! -- Highest Sales Order for Each Customer

Shows the biggest purchase made by each customer.


## Q3. Calculate total sales for each customer. (CTE)  

In [13]:
q3 = '''with sales_detials as(
select customer_id, 
sum(Sales) as total_sales 
from orders 
group by customer_id 
)

select o.order_id, o.order_date, o.customer_id, o.product_id, s.total_sales
from orders as o,
sales_detials as s'''

df_q3 = run_query(q3)
df_q3

,order_id,order_date,customer_id,product_id,total_sales
0,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,5563.560
1,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,1056.390
2,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,1790.512
3,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,5086.935
4,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,886.156
...,...,...,...,...,...
7924444,CA-2017-119914,5/4/2017,CC-12220,OFF-AP-10002684,2374.658
7924445,CA-2017-119914,5/4/2017,CC-12220,OFF-AP-10002684,5454.350
7924446,CA-2017-119914,5/4/2017,CC-12220,OFF-AP-10002684,6720.444
7924447,CA-2017-119914,5/4/2017,CC-12220,OFF-AP-10002684,8025.707


In [62]:
print("""BRIEF INSIGHTS!! -- Total Sales for Each Customer

Helps identify customers who contribute the most sales.""")

BRIEF INSIGHTS!! -- Total Sales for Each Customer

Helps identify customers who contribute the most sales.


## Q4. Find customers whose total sales are above average. (CTE + Subquery)  

In [14]:
q4 = '''with customer_sales as(
   select customer_id,
   sum(Sales) as total_sales 
   from orders
   group by customer_id
)

select cs.customer_id, c.customer_name, cs.total_sales
from customer_sales cs
join customers c
on cs.customer_id = c.customer_id
where cs.total_sales > (select avg(total_sales) from customer_sales)
'''

df_q4 = run_query(q4)
df_q4

,customer_id,customer_name,total_sales
0,BH-11710,Brosina Hoffman,6255.3510
1,IM-15070,Irene Maddox,4930.4740
2,PK-19075,Pete Kriz,8646.9340
3,TB-21520,Tracy Blumstein,4737.4860
4,MA-17560,Matt Abelman,4299.1610
...,...,...,...
289,SB-20185,Sarah Brown,6410.9960
290,TS-21655,Trudy Schmidt,3368.0940
291,VP-21760,Victoria Pisteka,3360.5260
292,CM-12715,Craig Molinari,3984.4524


In [61]:
print("""BRIEF INSIGHTS!! -- Customers with Above-Average Total Sales

These customers spend more than the average customer and are valuable to the business.""")

BRIEF INSIGHTS!! -- Customers with Above-Average Total Sales

These customers spend more than the average customer and are valuable to the business.


## Q5. Rank all customers based on total sales. (Window Function)  

In [15]:
q5 = '''with customer_sales as(
    select customer_id,
    sum(Sales) as total_sales
    from orders
    group by customer_id
)

select *,
rank() over(order by total_sales desc) as sales_rank
from customer_sales'''
df_q5 = run_query(q5)
df_q5

,customer_id,total_sales,sales_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


In [65]:
print("""BRIEF INSIGHTS!! -- Customer Ranking by Total Sales

Ranks customers from highest to lowest based on their spending.""")

BRIEF INSIGHTS!! -- Customer Ranking by Total Sales

Ranks customers from highest to lowest based on their spending.


## Q6. Assign row numbers to each order within a customer. (Window Function + PARTITION BY)  

In [18]:
q6 = ''' select *,
    row_number() over(partition by customer_id order by order_date) as row_rank 
    from orders '''
df_q6 = run_query(q6)
df_q6

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit,row_rank
0,CA-2015-121391,10/4/2015,10/7/2015,First Class,AA-10315,OFF-ST-10001590,26.960,2,0.0,7.0096,1
1,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,OFF-SU-10000151,3930.072,3,0.2,-786.0144,2
2,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,OFF-FA-10001332,2.304,1,0.2,0.7776,3
3,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,TEC-PH-10000895,431.976,3,0.2,32.3982,4
4,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,TEC-AC-10002857,41.720,7,0.2,5.7365,5
...,...,...,...,...,...,...,...,...,...,...,...
9988,CA-2016-152471,7/8/2016,7/8/2016,Same Day,ZD-21925,TEC-PH-10002824,823.960,5,0.2,51.4975,5
9989,CA-2016-152471,7/8/2016,7/8/2016,Same Day,ZD-21925,OFF-PA-10004965,15.984,2,0.2,4.9950,6
9990,CA-2014-143336,8/27/2014,9/1/2014,Second Class,ZD-21925,OFF-AR-10003056,8.560,2,0.0,2.4824,7
9991,CA-2014-143336,8/27/2014,9/1/2014,Second Class,ZD-21925,TEC-PH-10001949,213.480,3,0.2,16.0110,8


In [66]:
print("""BRIEF INSIGHTS!! -- Row Number for Each Customer's Orders

Shows the sequence in which each customer placed their orders.""")

BRIEF INSIGHTS!! -- Row Number for Each Customer's Orders

Shows the sequence in which each customer placed their orders.


## Q7. Display top 3 customers based on total sales. (Window Function)  

In [25]:
q7 = ''' with customer_sales as(
    select customer_id, sum(Sales) as total_sales 
    from orders
    group by customer_id
    ),

rank_order as(
    select * , rank() over(order by total_sales desc) as top_customers 
    from customer_sales 
)

select * 
from rank_order
where top_customers <= 3 '''

df_q7 = run_query(q7)
df_q7

,customer_id,total_sales,top_customers
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


In [67]:
print("""BRIEF INSIGHTS!! -- Top 3 Customers by Sales

Identifies the three customers who generated the highest revenue.""")

BRIEF INSIGHTS!! -- Top 3 Customers by Sales

Identifies the three customers who generated the highest revenue.


# STEP 5: Final Combined Query 

## Q. Write one final query that shows: Customer Name,Total Sales and Rank  

In [30]:
query = '''with customer_sales as (
       select customer_id, sum(Sales) as total_sales 
       from orders
       group by customer_id
       ),

customer_rank as(
       select *, rank() over(order by total_sales desc) as rank_of_customers
       from customer_sales
       )

select c.customer_name, cr.total_sales, cr.rank_of_customers
from customers c
join customer_rank cr
on c.customer_id = cr.customer_id
order by cr.rank_of_customers
'''

df_query = run_query(query)
df_query

,customer_name,total_sales,rank_of_customers
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [68]:
print("""BRIEF INSIGHTS!! -- Customer Name, Total Sales, and Rank

Provides a quick summary of customer performance and ranking.""")

BRIEF INSIGHTS!! -- Customer Name, Total Sales, and Rank

Provides a quick summary of customer performance and ranking.


# STEP 6: Mini Project - Customer Sales Insights 

## Q1. Who are the top 5 customers?  

In [51]:
q1 = '''with customer_sales as (
select customer_id,sum(Sales) total_sales from orders
group by customer_id
)

select cs.customer_id,c.customer_name,cs.total_sales ,dense_rank()over(order by cs.total_sales desc) as customer_rank 
from
customer_sales cs
inner join 
customers c
on cs.customer_id = c.customer_id
order by total_sales desc
limit 5'''

df_q1 = run_query(q1)
df_q1

,customer_id,customer_name,total_sales,customer_rank
0,SM-20320,Sean Miller,25043.050,1
1,TC-20980,Tamara Chand,19052.218,2
2,RB-19360,Raymond Buch,15117.339,3
3,TA-21385,Tom Ashbrook,14595.620,4
4,AB-10105,Adrian Barton,14473.571,5


In [70]:
print("""BRIEF INSIGHTS!! -- Top 5 Customers

Highlights the customers who contribute the most to sales.""")

BRIEF INSIGHTS!! -- Top 5 Customers

Highlights the customers who contribute the most to sales.


## Q2. Who are the bottom 5 customers?  

In [50]:
q2 = '''with customer_sales as (
select customer_id,sum(Sales) total_sales from orders
group by customer_id
)

select cs.customer_id,c.customer_name,cs.total_sales ,dense_rank()over(order by cs.total_sales desc) as customer_rank 
from
customer_sales cs
inner join 
customers c
on cs.customer_id = c.customer_id
order by total_sales asc
limit 5
'''

df_q2 = run_query(q2)
df_q2

,customer_id,customer_name,total_sales,customer_rank
0,TS-21085,Thais Sissman,4.833,793
1,LD-16855,Lela Donovan,5.304,792
2,CJ-11875,Carl Jackson,16.520,791
3,MG-18205,Mitch Gastineau,16.739,790
4,RS-19870,Roy Skaria,22.328,789


In [71]:
print("""BRIEF INSIGHTS!! -- Bottom 5 Customers

Shows customers with the lowest sales contribution.""")

BRIEF INSIGHTS!! -- Bottom 5 Customers

Shows customers with the lowest sales contribution.


## Q3. Which customers made only one order?  

In [34]:
q3 = '''
     select customer_id, count(order_id) as order_count
     from orders
     group by customer_id
     having count(order_id) = 1
     '''

df_q3 = run_query(q3)
df_q3

,customer_id,order_count
0,AO-10810,1
1,CJ-11875,1
2,JR-15700,1
3,LD-16855,1
4,RE-19405,1


In [72]:
print("""BRIEF INSIGHTS!! -- Customers with Only One Order

Identifies customers who purchased only once and may need re-engagement.""")

BRIEF INSIGHTS!! -- Customers with Only One Order

Identifies customers who purchased only once and may need re-engagement.


## Q4. Which customers have above-average sales?  

In [36]:
q4 = '''with customer_sales as(
   select customer_id,
   sum(Sales) as total_sales 
   from orders
   group by customer_id
   order by total_sales desc
)

select *
from customer_sales
where total_sales > (select avg(total_sales) from customer_sales)
'''

df_q4 = run_query(q4)
df_q4

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
289,JK-16120,2932.484
290,SW-20455,2921.544
291,ML-17410,2921.500
292,RD-19585,2912.894


In [73]:
print("""BRIEF INSIGHTS!! -- Customers with Above-Average Sales

These customers are important because they spend more than most others.""")

BRIEF INSIGHTS!! -- Customers with Above-Average Sales

These customers are important because they spend more than most others.


## Q5. What is the highest order value per customer? 

In [41]:
q5 = '''select customer_id,
        max(Sales) AS highest_order_value
        from orders
        group by customer_id
'''

df_q5 = run_query(q5)
df_q5



,customer_id,highest_order_value
0,AA-10315,3930.072
1,AA-10375,499.980
2,AA-10480,479.970
3,AA-10645,1323.900
4,AB-10015,341.960
...,...,...
788,XP-21865,337.088
789,YC-21895,2934.330
790,YS-21880,2793.528
791,ZC-21910,1516.200


In [74]:
print("""BRIEF INSIGHTS!! -- Highest Order Value per Customer

Shows the maximum amount spent by each customer in a single order.""")

BRIEF INSIGHTS!! -- Highest Order Value per Customer

Shows the maximum amount spent by each customer in a single order.
